# JS09 — Lab 3: Exact NN vs. HNSW

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** membandingkan **exact NN** (sklearn `NearestNeighbors`, brute) dengan **HNSW** (`hnswlib`) pada 1.000 titik acak 2D dengan `k = 5`, lalu menjelaskan perbedaan skala jarak keduanya.

**Prasyarat:** library `hnswlib` (dipasang otomatis oleh Langkah 1) dan `scikit-learn`. Tidak ada dataset eksternal.

> Jalankan di **Google Colab**: `Runtime` ▸ `Run all`. Notebook ini cukup memakai CPU, tidak perlu GPU.


# Lab 3 — Exact NN vs. HNSW

Modul: `JS09 › Lab 3 — Exact NN vs. HNSW`

Percobaan ini sama seperti Lab 2, tetapi pembandingnya **HNSW** (*Hierarchical Navigable Small World*) — ANN berbasis graf bertingkat. Data: 1.000 vektor 2D acak pada [0, 1), query `[0.5, 0.5]`, `k = 5`.


## Langkah 1 — Instal Library

In [ ]:
try:
    import hnswlib  # noqa: F401
except ImportError:
    !pip install -q hnswlib

## Langkah 2 — Import Library

In [ ]:
import hnswlib
import numpy as np
import time
from sklearn.neighbors import NearestNeighbors

## Langkah 3 — Buat Dataset Dummy

Modul **tidak** menetapkan seed di sini, jadi indeks hasil akan berbeda pada setiap run. Bila ingin hasil yang reproducible, tambahkan `np.random.seed(42)` sebelum baris `data = ...`, lalu jalankan ulang cell ini dan cell-cell berikutnya.

In [ ]:
# Build Dummy Dataset
num_elements = 1000
dim = 2
data = np.random.random((num_elements, dim)).astype(np.float32)

# Query point
query = np.array([[0.5, 0.5]], dtype=np.float32)
k = 5  # cari 5 tetangga terdekat

## Langkah 4 — Buat Model NN

Model exact memakai `algorithm='brute'` dan metrik Euclidean, yaitu pembanding "kebenaran" untuk HNSW.

In [ ]:
# Build NN Model
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(data)

# Compute time performace
start = time.time()
distances, indices = nn.kneighbors(query)
end = time.time()

print("=== Exact NN ===")
print("Indices:", indices)
print("Distances:", distances)
print("Waktu:", end - start, "s")

**Contoh output modul:**

```text
=== Exact NN ===
Indices: [[102 238 579 373 486]]
Distances: [[0.01328841 0.01816678 0.03427347 0.03446613 0.0375734 ]]
Waktu: 0.0011098384857177734 s
```

## Langkah 5 — Buat Model HNSW

Parameter penting:

| Parameter | Arti |
|---|---|
| `M = 16` | jumlah koneksi maksimum per node di graf |
| `ef_construction = 100` | lebar pencarian saat membangun graf (kualitas indeks) |
| `ef = 50` | lebar pencarian saat query — *trade-off* kecepatan vs akurasi |

Catatan penting: `space='l2'` pada hnswlib mengembalikan **jarak kuadrat**, bukan jarak Euclidean. Jadi angka `0.00017658` di output modul setara `sqrt(0.00017658) ≈ 0.01328834` — sama dengan jarak exact NN di atas. Yang berbeda hanya skalanya, bukan urutan tetangganya.

In [ ]:
# Initiate Index
p = hnswlib.Index(space='l2', dim=dim)  

# Define Max Elements
p.init_index(max_elements=num_elements, ef_construction=100, M=16)

# Add Data
p.add_items(data)

# Set searching parameter
p.set_ef(50)   # tradeoff speed vs accuracy

start = time.time()
labels, distances = p.knn_query(query, k=k)
end = time.time()

print("\n=== HNSW ===")
print("Indices:", labels)
print("Distances:", distances)
print("Waktu:", end - start, "s")

**Contoh output modul:**

```text
=== HNSW ===
Indices: [[102 238 579 373 486]]
Distances: [[0.00017658 0.00033003 0.00117467 0.00118791 0.00141176]]
Waktu: 0.00013875961303710938 s
```

Perhatikan: indeks HNSW **identik** dengan exact NN, sedangkan jaraknya berbeda skala (kuadrat). HNSW bisa sangat akurat pada dataset kecil karena `ef = 50` masih menjelajah sebagian besar graf.

In [ ]:
# [tambahan] Buktikan bahwa jarak hnswlib = kuadrat jarak Euclidean
import pandas as pd

d_exact, i_exact = nn.kneighbors(query)   # dihitung ulang: `distances` sudah tertimpa di Langkah 5

banding_jarak = pd.DataFrame({
    'index': i_exact[0],
    'sklearn (euclidean)': d_exact[0],
    'hnswlib (kuadrat)': distances[0],
    'akar(hnswlib)': np.sqrt(distances[0]),
}).round(6)
banding_jarak

### [tambahan] Pengaruh `ef` terhadap recall dan waktu

`ef` adalah parameter *trade-off* utama HNSW saat query. Nilai kecil = cepat tetapi recall turun; nilai besar = akurat tetapi lebih lambat. Diukur pada 100 query acak dengan `k = 5`.

In [ ]:
# [tambahan] Scan ef: recall@5 vs waktu
rng = np.random.default_rng(0)
qidx = rng.choice(num_elements, size=100, replace=False)
Q = data[qidx]

d_gt, i_gt = nn.kneighbors(Q)   # ground truth exact untuk 100 query

rows = []
for ef in [10, 25, 50, 100, 200]:
    p.set_ef(ef)
    start = time.time()
    labels_ef, _ = p.knn_query(Q, k=k)
    elapsed = (time.time() - start) * 1000 / len(Q)
    recall = np.mean([len(set(l) & set(g)) / k for l, g in zip(labels_ef, i_gt)])
    rows.append({'ef': ef, 'recall@5': recall, 'ms/query': elapsed})

hasil_ef = pd.DataFrame(rows).round(4)
hasil_ef

In [ ]:
# [tambahan] grafik: import matplotlib (modul Lab 3 tidak memakainya)
import matplotlib.pyplot as plt

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(hasil_ef['ef'], hasil_ef['recall@5'], marker='o', color='tab:green')
ax1.set_xlabel('ef (saat query)')
ax1.set_ylabel('Recall@5', color='tab:green')
ax1.set_xscale('log')
ax1.set_xticks(hasil_ef['ef'])
ax1.set_xticklabels(hasil_ef['ef'])

ax2 = ax1.twinx()
ax2.plot(hasil_ef['ef'], hasil_ef['ms/query'], marker='s', color='tab:red')
ax2.set_ylabel('Waktu query (ms/query)', color='tab:red')

plt.title('HNSW: pengaruh ef terhadap recall & waktu')
plt.tight_layout()
plt.show()

**Analisis:** TBD

*(Bandingkan `indices`/`labels` dan jarak exact vs HNSW. Apakah urutan tetangganya sama? Jelaskan mengapa `np.sqrt(distances)` menghasilkan angka yang sama dengan jarak sklearn, lalu jelaskan pengaruh `ef` dari tabel `[tambahan]`.)*

---
## Kesimpulan Lab 3

**Apakah hasil HNSW sama dengan exact NN pada percobaan ini?** TBD

**Mengapa jarak hnswlib dan sklearn berbeda angka?** TBD

**Pengaruh `ef` terhadap recall dan waktu:** TBD

**Kesimpulan umum:** TBD